# Extract metadata + biomarker columns from both Excel sheets

This notebook reads the DELCODE baseline and follow-up Excel files, selects
relevant columns, and produces several output CSVs:

| Output | Description |
|--------|-------------|
| `*_selected_columns.csv` | All selected columns with MultiIndex group headers |
| `*_renamed_columns.csv` | Core subset renamed for downstream use (no `_CV` cols) |
| `*_baseline_followup_*.csv` | Baseline + follow-up combined, sorted by subject + date |

All `visdate` values are normalised to **DD-MM-YYYY** format.

In [1]:
import pandas as pd
import os
import numpy as np

# ── Input paths ──────────────────────────────────────────────────────────────

BASE = "/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION"

BL_EXCEL = "/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/study_metadata/Antrag_462_Ruat_DysConnectivity Index_20240320_DELCODE_Baseline_repseudonymisiert.xlsx"
FU_EXCEL = "/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/study_metadata/Antrag_462_Ruat_DysConnectivity Index_20240320DELCODE_Follow-up_repseudonymisiert.xlsx"

# ── Output directory ─────────────────────────────────────────────────────────
OUT_DIR = os.path.join(BASE, "data", "intermediate", "combined")
os.makedirs(OUT_DIR, exist_ok=True)

BL_SELECTED_PATH   = os.path.join(OUT_DIR, "baseline_selected.csv")
FU_SELECTED_PATH   = os.path.join(OUT_DIR, "followup_selected.csv")
BL_RENAMED_PATH    = os.path.join(OUT_DIR, "baseline_renamed.csv")
FU_RENAMED_PATH    = os.path.join(OUT_DIR, "followup_renamed.csv")
COMBINED_RENAMED   = os.path.join(BASE, "data", "intermediate", "combined", "all_visits.csv")
COMBINED_SELECTED   = os.path.join(BASE, "data", "intermediate", "combined", "all_visits_selected.csv")

# ── Date format for all outputs ─────────────────────────────────────────────
DATE_FMT = "%d-%m-%Y"

# ── Valid prmdiag values (exclude 100 = Angehörige von AD-Probanden) ────────
# 0: Gesunde Kontrollprobanden, 1: SCD, 2: MCI, 5: AD
VALID_PRMDIAG = {0, 1, 2, 5}

In [2]:
# ── Shared helpers ───────────────────────────────────────────────────────────

def read_excel_with_header(path):
    """Read an Excel file where row 0 = section headers, row 1 = column names."""
    raw = pd.read_excel(path, header=None)
    header = raw.iloc[1].values
    df = raw.iloc[2:].copy()
    df.columns = header
    df = df.reset_index(drop=True)
    print(f"Loaded {path.split('/')[-1]}: raw {raw.shape} → data {df.shape}")
    return df


def select_columns(df, columns_by_index):
    """Select columns by positional index and rename them."""
    out = df.iloc[:, list(columns_by_index.keys())].copy()
    out.columns = list(columns_by_index.values())
    return out


def normalise_visdate(df, col="visdate"):
    """Convert visdate column to DD-MM-YYYY string format."""
    df[col] = pd.to_datetime(df[col], errors="coerce").dt.strftime(DATE_FMT)
    return df


# ── Shared group-header definitions ──────────────────────────────────────────
# These are identical between baseline and follow-up (except for extras).
SHARED_GROUPS = {
    "MMSE": ["mmstot", "mmsort", "mmsorp", "mmsreg", "mmsac", "mmsrl", "mmslng", "mmsdw"],
    "GDS": ["gdstot", "gdsframe"] + [f"gds{i}" for i in range(1, 16)],
    "FAQ": ["faqtot"] + [f"faq{i}" for i in range(1, 11)] + ["primsrc", "prmsoth"],
    "CDR": ["cdrtot", "cdrglobal"] + [f"cdr010{i}" for i in range(1, 7)],
    "ApoE": ["ApoE"],
    "Neurodegenerationsmarker": [
        "Abeta38", "Abeta40", "Abeta42",
        "totaltau", "phosphotau181",
        "ratio_Abeta42_40", "ratio_Abeta42_phosphotau181",
    ],
    "ADAS-Cog": ["AD11SUM", "AD13SUM"],
    "Abeta 40/42 (MSD)": ["AB40-MSD pg/ml", "AB42-MSD pg/ml", "Ratio42_40-MSD"],
}


def build_group_map(groups_dict):
    """Build {column: group_name} from a dict of {group: [columns]}."""
    gm = {}
    for grp, cols in groups_dict.items():
        for c in cols:
            gm[c] = grp
    return gm


def save_with_multiindex(df, group_map, path):
    """Add MultiIndex group header and save to CSV."""
    group_row = [group_map[c] for c in df.columns]
    df.columns = pd.MultiIndex.from_arrays([group_row, df.columns])
    df.to_csv(path, index=False)
    print(f"Saved: {path}  ({len(df)} rows × {len(df.columns)} cols)")
    print(f"Read back with: pd.read_csv('{path}', header=[0, 1])")
    return df


def compute_age_and_rename(df, has_visnam=False):
    """Select core columns, compute age, rename Pseudonym → Repseudonym."""
    keep = ["Pseudonym"]
    if has_visnam:
        keep.append("visnam")
    keep += [
        "visdate", "sex", "brthdat", "prmdiag", "mmstot", "cdrtot", "cdrglobal",
        "Abeta38", "Abeta40", "Abeta42",
        "totaltau", "phosphotau181",
        "ratio_Abeta42_40", "ratio_Abeta42_phosphotau181",
    ]
    out = df[keep].copy()
    out = out.rename(columns={"Pseudonym": "Repseudonym"})

    # Compute age (parse dates before visdate has been reformatted to string)
    visdt = pd.to_datetime(out["visdate"], dayfirst=True, errors="coerce")
    bthdt = pd.to_datetime(out["brthdat"], errors="coerce")
    out["age"] = ((visdt - bthdt).dt.days / 365.25).round(2)
    out = out.drop(columns=["brthdat"])

    # Normalise visdate to DD-MM-YYYY
    out["visdate"] = visdt.dt.strftime(DATE_FMT)

    # Reorder
    order = ["Repseudonym"]
    if has_visnam:
        order.append("visnam")
    order += [
        "visdate", "sex", "age", "prmdiag", "mmstot", "cdrtot", "cdrglobal",
        "Abeta38", "Abeta40", "Abeta42",
        "totaltau", "phosphotau181",
        "ratio_Abeta42_40", "ratio_Abeta42_phosphotau181",
    ]
    return out[order]


def combine_baseline_followup(df_bl, df_fu, out_path, label):
    """Tag, union, sort, and save baseline + follow-up."""
    bl = df_bl.copy(); bl["file"] = "BASELINE"
    fu = df_fu.copy(); fu["file"] = "FOLLOWUP"

    bl_cols = [c for c in bl.columns if c != "file"]
    fu_extra = [c for c in fu.columns if c not in bl_cols and c != "file"]
    ordered = ["file"] + bl_cols + fu_extra

    bl = bl.reindex(columns=ordered)
    fu = fu.reindex(columns=ordered)
    combined = pd.concat([bl, fu], ignore_index=True)

    subj = "Repseudonym" if "Repseudonym" in combined.columns else "Pseudonym"
    combined["_dt"] = pd.to_datetime(combined["visdate"], dayfirst=True, errors="coerce")
    combined["_fo"] = combined["file"].map({"BASELINE": 0, "FOLLOWUP": 1}).fillna(9)
    combined = combined.sort_values([subj, "_dt", "_fo"], kind="mergesort").reset_index(drop=True)
    combined = combined.drop(columns=["_dt", "_fo"])
    combined.to_csv(out_path, index=False)

    print(f"Saved {label}: {out_path}  ({len(combined)} rows × {len(combined.columns)} cols)")
    print(combined["file"].value_counts().to_string())
    return combined

---
## 1. Baseline Data Extraction

In [3]:
df_bl_raw = read_excel_with_header(BL_EXCEL)

bl_columns = {
    # Basisdaten
    0: "Pseudonym", 3: "visdate", 6: "sex", 7: "brthdat", 17: "prmdiag",
    # MMSE
    165: "mmstot", 166: "mmsort", 167: "mmsorp", 168: "mmsreg",
    169: "mmsac", 170: "mmsrl", 171: "mmslng", 172: "mmsdw",
    # GDS
    173: "gdstot", 174: "gdsframe",
    175: "gds1", 176: "gds2", 177: "gds3", 178: "gds4", 179: "gds5",
    180: "gds6", 181: "gds7", 182: "gds8", 183: "gds9", 184: "gds10",
    185: "gds11", 186: "gds12", 187: "gds13", 188: "gds14", 189: "gds15",
    # FAQ
    248: "faqtot",
    249: "faq1", 250: "faq2", 251: "faq3", 252: "faq4", 253: "faq5",
    254: "faq6", 255: "faq7", 256: "faq8", 257: "faq9", 258: "faq10",
    259: "primsrc", 260: "prmsoth",
    # CDR
    261: "cdrtot", 262: "cdrglobal",
    263: "cdr0101", 264: "cdr0102", 265: "cdr0103",
    266: "cdr0104", 267: "cdr0105", 268: "cdr0106",
    # ApoE
    777: "ApoE",
    # Neurodegenerationsmarker (CSF biomarkers) — _CV columns dropped
    778: "Abeta38", 780: "Abeta40", 782: "Abeta42",
    784: "totaltau", 786: "phosphotau181",
    788: "ratio_Abeta42_40", 789: "ratio_Abeta42_phosphotau181",
    # ADAS-Cog
    790: "AD11SUM", 791: "AD13SUM",
    # Abeta 40/42 (MSD)
    822: "AB40-MSD pg/ml", 823: "AB42-MSD pg/ml", 824: "Ratio42_40-MSD",
}

df_bl = select_columns(df_bl_raw, bl_columns)
df_bl = normalise_visdate(df_bl)  # → DD-MM-YYYY
print(f"Baseline selected: {df_bl.shape}")
print(f"Sample visdate: {df_bl['visdate'].iloc[0]}")

Loaded Antrag_462_Ruat_DysConnectivity Index_20240320_DELCODE_Baseline_repseudonymisiert.xlsx: raw (939, 825) → data (937, 825)
Baseline selected: (937, 64)
Sample visdate: 05-01-2016


In [4]:
bl_groups = {
    "Basisdaten": ["Pseudonym", "visdate", "sex", "brthdat", "prmdiag"],
    **SHARED_GROUPS,
}
save_with_multiindex(df_bl, build_group_map(bl_groups), BL_SELECTED_PATH)

Saved: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/combined/baseline_selected.csv  (937 rows × 64 cols)
Read back with: pd.read_csv('/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/combined/baseline_selected.csv', header=[0, 1])


Basisdaten                                    MMSE                       \
     Pseudonym     visdate sex  brthdat prmdiag mmstot mmsort mmsorp mmsreg   
0    93f563391  05-01-2016   m  1953-03       0     29      5      5      3   
1    043ba1305  05-12-2017   m  1939-12       1     29      5      4      3   
2    157dec7d5  08-09-2015   m  1945-11       0     30      5      5      3   
3    5598d51dd  12-07-2018   f  1943-01       2     28      5      4      3   
4    8a025f647  22-12-2014   f  1936-12       0     29      5      5      3   
..         ...         ...  ..      ...     ...    ...    ...    ...    ...   
932  dbd2ab28d  20-08-2018   f  1944-02     100     30      5      5      3   
933  312c25423  10-01-2018   f  1938-05       2     28      5      5      3   
934  0449f2db9  26-03-2015   m  1949-02       0     29      5      5      3   
935  5fabfb164  19-01-2017   f  1946-12       0     30      5      5      3   
936  7a14bdf5d  04-07-2017   f  1942-05       5     22      5      4      3   

           ... Neurodegenerationsmarker                             \
    mmsac  ...                  Abeta42     totaltau phosphotau181   
0       5  ...               440.539668   561.004123          64.7   
1       5  ...               323.331467   415.636471        59.727   
2       5  ...                      NaN          NaN           NaN   
3       5  ...               267.630106  1146.874214        160.87   
4       5  ...                861.96831   295.001296          32.4   
..    ...  ...                      ...          ...           ...   
932     5  ...                      NaN          NaN           NaN   
933     5  ...                      NaN          NaN           NaN   
934     5  ...               702.374585   234.747744          32.5   
935     5  ...              1044.698554   509.702049        56.928   
936     4  ...               354.544913   504.434206         62.05   

                                                   ADAS-Cog             \
    ratio_Abeta42_40 ratio_Abeta42_phosphotau181    AD11SUM    AD13SUM   
0           0.042063                    6.808959   2.333333   4.333333   
1           0.063097                    5.413489   5.666667  10.666667   
2                NaN                         NaN          2          5   
3           0.044748                    1.663642         10         18   
4           0.117943                    26.60396   3.333333   4.333333   
..               ...                         ...        ...        ...   
932              NaN                         NaN   4.666667   6.666667   
933              NaN                         NaN   9.666667  18.666667   
934         0.110477                   21.611526   3.333333   5.333333   
935         0.108799                   18.351225   2.333333   3.333333   
936         0.052558                    5.713858  15.333333  23.333333   

    Abeta 40/42 (MSD)                                        
       AB40-MSD pg/ml    AB42-MSD pg/ml      Ratio42_40-MSD  
0    56.8838301059518  5.79788519606056   0.101932859067638  
1    39.6242349725417    3.642827269388  0.0918812591768559  
2    83.8188411024443  8.85031854090964   0.105585319211733  
3    94.0169370265133  10.9500950939099   0.116513561448998  
4    72.7577659234397  7.80686346894039   0.107230061981809  
..                ...               ...                 ...  
932  105.854962795727  9.68613890970355  0.0915799764155948  
933  120.515764301618  10.9558836201957  0.0909174413973317  
934  85.4385456576107  10.8026347898469   0.126581606333476  
935  50.8725761613296  6.46281127443602   0.127067751254023  
936  54.1268480584377  5.45829393580857   0.100870070029685  

[937 rows x 64 columns]

---
## 2. Follow-Up Data Extraction

In [ ]:
df_fu_raw = read_excel_with_header(FU_EXCEL)

fu_columns = {
    # Basisdaten
    0: "Pseudonym", 2: "visnam", 3: "visdate", 7: "sex", 8: "brthdat", 18: "prmdiag",
    # MMSE
    577: "mmstot", 578: "mmsort", 579: "mmsorp", 580: "mmsreg",
    581: "mmsac", 582: "mmsrl", 583: "mmslng", 584: "mmsdw",
    # GDS
    326: "gdstot", 327: "gdsframe",
    328: "gds1", 329: "gds2", 330: "gds3", 331: "gds4", 332: "gds5",
    333: "gds6", 334: "gds7", 335: "gds8", 336: "gds9", 337: "gds10",
    338: "gds11", 339: "gds12", 340: "gds13", 341: "gds14", 342: "gds15",
    # FAQ
    282: "faqtot",
    283: "faq1", 284: "faq2", 285: "faq3", 286: "faq4", 287: "faq5",
    288: "faq6", 289: "faq7", 290: "faq8", 291: "faq9", 292: "faq10",
    610: "primsrc", 611: "prmsoth",
    # CDR
    167: "cdrtot", 168: "cdrglobal",
    169: "cdr0101", 170: "cdr0102", 171: "cdr0103",
    172: "cdr0104", 173: "cdr0105", 174: "cdr0106",
    # Neurodegenerationsmarker — _CV columns dropped
    941: "Abeta38", 943: "Abeta40", 945: "Abeta42",
    947: "totaltau", 949: "phosphotau181",
    951: "ratio_Abeta42_40", 952: "ratio_Abeta42_phosphotau181",
    # ADAS-Cog
    938: "AD11SUM", 939: "AD13SUM",
    # Follow-up extras
    940: "pacc5", 953: "ND_Assay", 954: "Kommentar",
}

df_fu = select_columns(df_fu_raw, fu_columns)
df_fu = normalise_visdate(df_fu)  # → DD-MM-YYYY

# Add empty columns for baseline-only fields
for col in ["ApoE", "AB40-MSD pg/ml", "AB42-MSD pg/ml", "Ratio42_40-MSD"]:
    df_fu[col] = np.nan

print(f"Follow-up selected: {df_fu.shape}")
print(f"Sample visdate: {df_fu['visdate'].iloc[0]}")
print(f"Unique visits: {df_fu['visnam'].unique()}")

In [ ]:
fu_groups = {
    "Basisdaten": ["Pseudonym", "visnam", "visdate", "sex", "brthdat", "prmdiag"],
    **SHARED_GROUPS,
    "NPT_Scores": ["pacc5"],
    "Follow-Up Extra": ["ND_Assay", "Kommentar"],
}
save_with_multiindex(df_fu, build_group_map(fu_groups), FU_SELECTED_PATH)

Saved: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/combined/followup_selected.csv  (4077 rows × 68 cols)
Read back with: pd.read_csv('/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/combined/followup_selected.csv', header=[0, 1])


Basisdaten                                                 MMSE         \
      Pseudonym       visnam     visdate sex  brthdat prmdiag mmstot mmsort   
0     93f563391  Follow-up 1  18-01-2017   m  1953-03       0     29      5   
1     93f563391  Follow-up 2  16-01-2018   m  1953-03       0     29      5   
2     93f563391  Follow-up 3  16-01-2019   m  1953-03       0     30      5   
3     93f563391  Follow-up 4  15-01-2020   m  1953-03       0     30      5   
4     93f563391  Follow-up 5  11-03-2021   m  1953-03       0     29      5   
...         ...          ...         ...  ..      ...     ...    ...    ...   
4072  5fabfb164  Follow-up 7  08-01-2024   f  1946-12       0     30      5   
4073  7a14bdf5d  Follow-up 1  16-07-2018   f  1942-05       5     20      4   
4074  7a14bdf5d  Follow-up 2  19-08-2019   f  1942-05       5     21      3   
4075  7a14bdf5d  Follow-up 3  03-08-2020   f  1942-05       5     15      2   
4076  7a14bdf5d  Follow-up 4  23-11-2021   f  1942-05       5      9      0   

                    ...    Neurodegenerationsmarker ADAS-Cog          \
     mmsorp mmsreg  ... ratio_Abeta42_phosphotau181  AD11SUM AD13SUM   
0         5      3  ...                         NaN        2       5   
1         5      3  ...                    6.835696        1       2   
2         5      3  ...                         NaN        5      10   
3         5      3  ...                    4.801115     4.33   10.33   
4         5      3  ...                         NaN        5      12   
...     ...    ...  ...                         ...      ...     ...   
4072      5      3  ...                         NaN     3.67    5.67   
4073      4      3  ...                         NaN    20.67   31.67   
4074      3      3  ...                    4.882587      NaN     NaN   
4075      3      3  ...                         NaN      NaN     NaN   
4076      1      3  ...                         NaN      NaN     NaN   

     NPT_Scores Follow-Up Extra           ApoE Abeta 40/42 (MSD)  \
          pacc5        ND_Assay Kommentar ApoE    AB40-MSD pg/ml   
0         -0.04             NaN       NaN  NaN               NaN   
1          0.04               0       NaN  NaN               NaN   
2         -0.22             NaN       NaN  NaN               NaN   
3         -0.05               0       NaN  NaN               NaN   
4         -1.01             NaN       NaN  NaN               NaN   
...         ...             ...       ...  ...               ...   
4072       1.22             NaN       NaN  NaN               NaN   
4073       -4.2             NaN       NaN  NaN               NaN   
4074        NaN               0       NaN  NaN               NaN   
4075        NaN             NaN       NaN  NaN               NaN   
4076        NaN             NaN       NaN  NaN               NaN   

                                    
     AB42-MSD pg/ml Ratio42_40-MSD  
0               NaN            NaN  
1               NaN            NaN  
2               NaN            NaN  
3               NaN            NaN  
4               NaN            NaN  
...             ...            ...  
4072            NaN            NaN  
4073            NaN            NaN  
4074            NaN            NaN  
4075            NaN            NaN  
4076            NaN            NaN  

[4077 rows x 68 columns]

---
## 3. Renamed Column Subset
Select core columns for downstream classification, compute age, rename
`Pseudonym` → `Repseudonym`, drop `_CV` columns, normalise `visdate` to
`DD-MM-YYYY`, and filter out relatives (`prmdiag=100`).

In [ ]:
# Re-read selected CSVs as flat DataFrames
df_bl_flat = pd.read_csv(BL_SELECTED_PATH, header=[0, 1])
df_bl_flat.columns = df_bl_flat.columns.get_level_values(1)

df_fu_flat = pd.read_csv(FU_SELECTED_PATH, header=[0, 1])
df_fu_flat.columns = df_fu_flat.columns.get_level_values(1)

# Compute age + rename (shared helper — also normalises visdate)
df_bl_renamed = compute_age_and_rename(df_bl_flat, has_visnam=False)
df_fu_renamed = compute_age_and_rename(df_fu_flat, has_visnam=True)

# Filter prmdiag
bl_n = len(df_bl_renamed)
df_bl_renamed = df_bl_renamed[df_bl_renamed["prmdiag"].isin(VALID_PRMDIAG)].reset_index(drop=True)
fu_n = len(df_fu_renamed)
df_fu_renamed = df_fu_renamed[df_fu_renamed["prmdiag"].isin(VALID_PRMDIAG)].reset_index(drop=True)

print(f"Baseline:  {bl_n} → {len(df_bl_renamed)} (removed {bl_n - len(df_bl_renamed)} relatives)")
print(f"Follow-up: {fu_n} → {len(df_fu_renamed)} (removed {fu_n - len(df_fu_renamed)} relatives)")

# Save
df_bl_renamed.to_csv(BL_RENAMED_PATH, index=False)
df_fu_renamed.to_csv(FU_RENAMED_PATH, index=False)

print(f"\nBaseline prmdiag: {df_bl_renamed['prmdiag'].value_counts().sort_index().to_dict()}")
print(f"Follow-up prmdiag: {df_fu_renamed['prmdiag'].value_counts().sort_index().to_dict()}")
print(f"Sample visdate: {df_bl_renamed['visdate'].iloc[0]}")
df_bl_renamed.head(3)

Baseline:  937 → 866 (removed 71 relatives)
Follow-up: 4077 → 3739 (removed 338 relatives)

Baseline prmdiag: {0: 221, 1: 381, 2: 157, 5: 107}
Follow-up prmdiag: {0: 1175, 1: 1785, 2: 557, 5: 222}
Sample visdate: 05-01-2016


,Repseudonym,visdate,sex,age,prmdiag,mmstot,cdrtot,cdrglobal,Abeta38,Abeta40,Abeta42,totaltau,phosphotau181,ratio_Abeta42_40,ratio_Abeta42_phosphotau181
0,93f563391,05-01-2016,m,62.85,0,29,0,0,2853.300127,10473.336298,440.539668,561.004123,64.700,0.042063,6.808959
1,043ba1305,05-12-2017,m,78.01,1,29,3,0.5,2014.163435,5124.315052,323.331467,415.636471,59.727,0.063097,5.413489
2,157dec7d5,08-09-2015,m,69.85,0,30,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## 4. Combine Baseline + Follow-up
Create two combined tables with a `file` column (`BASELINE` / `FOLLOWUP`),
sorted by subject then visit date.

In [ ]:
# Combined renamed
comb_renamed = combine_baseline_followup(df_bl_renamed, df_fu_renamed, COMBINED_RENAMED, "combined_renamed")
print()

# Combined selected (re-read flat, filter prmdiag)
df_bl_sel = pd.read_csv(BL_SELECTED_PATH, header=[0, 1])
df_bl_sel.columns = df_bl_sel.columns.get_level_values(1)
df_fu_sel = pd.read_csv(FU_SELECTED_PATH, header=[0, 1])
df_fu_sel.columns = df_fu_sel.columns.get_level_values(1)

if "prmdiag" in df_bl_sel.columns:
    df_bl_sel = df_bl_sel[df_bl_sel["prmdiag"].isin(VALID_PRMDIAG)].reset_index(drop=True)
if "prmdiag" in df_fu_sel.columns:
    df_fu_sel = df_fu_sel[df_fu_sel["prmdiag"].isin(VALID_PRMDIAG)].reset_index(drop=True)

comb_selected = combine_baseline_followup(df_bl_sel, df_fu_sel, COMBINED_SELECTED, "combined_selected")

# Preview
print("\nPreview (renamed):")
display(comb_renamed[[c for c in ["Repseudonym", "visnam", "visdate", "file", "prmdiag"] if c in comb_renamed.columns]].head(10))

Saved combined_renamed: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/combined/all_visits.csv  (4605 rows × 17 cols)
file
FOLLOWUP    3739
BASELINE     866

Saved combined_selected: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/combined/all_visits_selected.csv  (4605 rows × 69 cols)
file
FOLLOWUP    3739
BASELINE     866

Preview (renamed):


,Repseudonym,visnam,visdate,file,prmdiag
0,011d501d1,NaN,01-10-2015,BASELINE,0
1,011d501d1,Follow-up 1,12-10-2016,FOLLOWUP,0
2,011d501d1,Follow-up 2,20-09-2017,FOLLOWUP,0
3,011d501d1,Follow-up 3,27-09-2018,FOLLOWUP,0
4,011d501d1,Follow-up 5,28-10-2020,FOLLOWUP,0
5,011d501d1,Follow-up 6,01-10-2021,FOLLOWUP,0
6,011d501d1,Follow-up 7,28-10-2022,FOLLOWUP,0
7,011d501d1,Follow-up 8,29-09-2023,FOLLOWUP,0
8,01490270a,NaN,27-02-2017,BASELINE,5
9,01490270a,Follow-up 1,07-02-2018,FOLLOWUP,5


---
## 5. Merge Scan Dates
Match resting-state scan dates to clinical visits using visit code mapping:
M0→Baseline, M12→Follow-up 1, M24→Follow-up 2, M36→Follow-up 3,
M48→Follow-up 4, M60→Follow-up 5. Adds `scan_date` and `scan_year` columns.

In [ ]:
# ── Merge scan dates into combined renamed CSV (visit code mapping) ────
import pandas as pd
import numpy as np

SCAN_XLSX = os.path.join(BASE, 'data', 'study_metadata', 'restingstate_scan_dates_M0_M60.xlsx')

# Visit code → visnam mapping
VISIT_MAP = {
    'M0':  'BASELINE',
    'M12': 'Follow-up 1',
    'M24': 'Follow-up 2',
    'M36': 'Follow-up 3',
    'M48': 'Follow-up 4',
    'M60': 'Follow-up 5',
}

# Load scan dates (T1_01 sheet)
scans = pd.read_excel(SCAN_XLSX, sheet_name='T1_01')
scans = scans[['pseudonym', 'visit', 'scan_date']].dropna(subset=['scan_date']).copy()
scans['scan_date'] = pd.to_datetime(scans['scan_date'], errors='coerce')
scans = scans.dropna(subset=['scan_date'])
scans['scan_date'] = scans['scan_date'].dt.strftime('%d-%m-%Y')
scans['_merge_key'] = scans['visit'].map(VISIT_MAP)
scans = scans.rename(columns={'pseudonym': 'Repseudonym', 'visit': 'scan_year'})
print(f'Scan dates loaded: {len(scans)} rows, {scans["Repseudonym"].nunique()} patients')

# Re-read the combined renamed CSV
comb = pd.read_csv(COMBINED_RENAMED)
comb['_merge_key'] = comb['visnam'].fillna('BASELINE')

# Merge on (Repseudonym, _merge_key)
comb = comb.merge(
    scans[['Repseudonym', '_merge_key', 'scan_date', 'scan_year']],
    on=['Repseudonym', '_merge_key'],
    how='left'
)
comb = comb.drop(columns=['_merge_key'])

# Save
comb.to_csv(COMBINED_RENAMED, index=False)

matched = comb['scan_date'].notna().sum()
total = len(comb)
print(f'\nScan date merge complete (visit code mapping):')
print(f'  Matched: {matched} / {total} rows ({100*matched/total:.1f}%)')
print(f'  Unmatched: {total - matched} (Follow-up 6+, Tel visits, or missing visits)')
comb[['Repseudonym', 'visdate', 'scan_date', 'scan_year', 'file']].head(10)


Scan dates loaded: 3321 rows, 964 patients

Scan date merge complete (visit code mapping):
  Matched: 2943 / 4605 rows (63.9%)
  Unmatched: 1662 (Follow-up 6+, Tel visits, or missing visits)


,Repseudonym,visdate,scan_date,scan_year,file
0,011d501d1,01-10-2015,08-10-2015,M0,BASELINE
1,011d501d1,12-10-2016,11-11-2016,M12,FOLLOWUP
2,011d501d1,20-09-2017,NaN,NaN,FOLLOWUP
3,011d501d1,27-09-2018,NaN,NaN,FOLLOWUP
4,011d501d1,28-10-2020,NaN,NaN,FOLLOWUP
5,011d501d1,01-10-2021,NaN,NaN,FOLLOWUP
6,011d501d1,28-10-2022,NaN,NaN,FOLLOWUP
7,011d501d1,29-09-2023,NaN,NaN,FOLLOWUP
8,01490270a,27-02-2017,01-03-2017,M0,BASELINE
9,01490270a,07-02-2018,07-02-2018,M12,FOLLOWUP
